In [ ]:
import cv2
import numpy as np
import streamlit as st
from ultralytics import YOLO

st.set_page_config(page_title="Yolo Object Detection",layout="centered")
st.title("Yolo Object Detection")

@st.cache_resource
def load_yolo_model():
    return YOLO("yolov8l.pt")

try:
    model = load_yolo_model()
    all_classes = sorted(list(model.names.values()))
except Exception as e:
    st.error(f"Failed to Load Yolo Model, An Error has Occured = {e}")
    st.stop()

selected_class = st.selectbox(
    "Choose your Target Class :",
    options=all_classes,
    index=all_classes.index("car") if "car" in all_classes else 0
)

if "uploader_key" not in st.session_state:
    st.session_state.uploader_key = 0

uploaded_file = st.file_uploader(
    "Upload Image", 
    type=["jpg", "jpeg", "png"], 
    key=f"uploader_{st.session_state.uploader_key}"
)

if uploaded_file is not None:
    file_bytes = np.asarray(bytearray(uploaded_file.read()), dtype=np.uint8)
    img = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    
    imgResult = img.copy()
    h, w, c = img.shape
    
    results = model(img)
    
    detected_coordinates = []
    
    for result in results:
        boxes = result.boxes
        for box in boxes:
            cls_id = int(box.cls)
            class_name = model.names[cls_id]
            
            if class_name == selected_class:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                
                confidence = float(box.conf[0]) * 100
                
                center_x = (x1 + x2) // 2
                center_y = (y1 + y2) // 2
                detected_coordinates.append((center_x, center_y))
                
                cv2.rectangle(imgResult, (x1, y1), (x2, y2), color=(255, 0, 255), thickness=4)
                
                label_text = f"Confidence = {confidence:.1f} | Class = {class_name.title()}"
                cv2.putText(imgResult, label_text, (x1, y1 - 10), 
                            cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.7, color=(255, 0, 255), thickness=2)

    imgRGB = cv2.cvtColor(imgResult, cv2.COLOR_BGR2RGB)
    
    st.markdown("---")
    st.image(imgRGB,  use_container_width=True)
    
    plural_header = f"{selected_class.title()}s" if not selected_class.endswith('s') else selected_class.title()
    st.markdown(f"{plural_header} =")
    
    if detected_coordinates:
        for idx, (cx, cy) in enumerate(detected_coordinates, start=1):
            st.write(f"● ( {cx} , {cy} )")
    else:
        st.write("No Matching Objects Detected in the Image")
        
    st.markdown("---")
    if st.button("Clear and Upload Another Image", type="primary"):
        st.session_state.uploader_key += 1
        st.rerun()

else:
    st.info("Please Upload a Valid Image")